# Notebook 1: Sentinel-1 preprocessing and EDA

This notebook converts the building labels created in the shared notebook 0 into
model-ready Sentinel-1 patches. It performs four tasks:

1. Export one pre-war VV/VH composite per city and one post-event composite
   for each labelled or temporal-offset date.
2. Build one canonical building table per city.
3. Cut row-aligned, memory-mapped image patches and validity masks.
4. Summarize the processed data and inspect the SAR signal in Gaza and the
   registered holdout cities.

`PREP` in `pipeline.py` defines the preprocessing settings, while
`CITY_REGISTRY` defines the cities and assessment dates. Processed filenames
encode the settings that affect their contents. Label files present on disk
but absent from `CITY_REGISTRY` are reported and left unused.

Gaza is the development city. Notebook 2 divides it into spatial train,
stack, validation and test bands. Raqqa, Mosul, Chernihiv and Rubizhne are
city-level holdouts and do not enter those development splits.

The pre-war composite is stored once per city because it is shared by all
assessment dates. Post-event composites use the closest Sentinel-1
acquisition day or days within the configured search radius. Same-day slices
are mosaicked before an acquisition day is counted. Optional fallback dates
fill only native nodata pixels and never replace valid primary pixels.

Temporal-offset composites provide neighbouring observations for the
second-stage model. They are unlabelled and are not used to fit the CNN. A
12-day offset matches Sentinel-1's repeat interval for a fixed relative orbit.


## 1. Setup

The active paths and preprocessing settings are printed below. The main
parameters are:

* `positive_damage_codes`: UNOSAT classes represented by `class=1`
* `post_search_days`: maximum distance from a target date to a candidate scene
* `post_n_labelled`: acquisition days in a labelled post composite
* `post_fallback_dates`: additional days used only for native nodata
* `temporal`: spacing and extent of the unlabelled offset series

`prep_tag()` includes the label definition, patch geometry, sample size and
sensor. A change to those settings creates new building and patch filenames.
Raster filenames separately encode the pre and post compositing variants.

The nearest-scene rule searches before and after each target date.
`check_city_windows()` reports searches that reach the pre-war period or
overlap a neighbouring assessment search.

A labelled composite normally uses one acquisition day. Section 2c compares
one and two acquisition days as a speckle-reduction diagnostic. Temporal
offsets remain at one acquisition day so neighbouring points on the 12-day
grid do not share scenes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q geedim geemap


In [ ]:
import os
import sys

PROJECT_ROOT = "/content/drive/MyDrive/War-Damage-Detection"
SENTINEL_DIR = os.path.join(PROJECT_ROOT, "sentinel_sar")
for path in (PROJECT_ROOT, SENTINEL_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
import ee
import geemap

from sentinel_sar.pipeline import (PREP, CITY_REGISTRY, LABELED_FOOTPRINT_DIR, PROCESSED_DIR,
                      RASTER_DIR, S1_EARLIEST,
                      load_labels, list_label_dates, sample_buildings,
                      prep_tag,
                      pre_raster_path, post_raster_path,
                      post_primary_raster_path,
                      read_export_meta, write_export_meta,
                      buildings_path, pre_patch_paths, post_patch_paths,
                      open_patch_writer, temporal_offsets, offset_dates,
                      temporal_scene_count, post_jobs,
                      load_city, post_arrays, usable_mask,
                      summarize_processed,
                      city_role, development_cities, held_out_cities,
                      check_city_windows, inventory, inventory_summary,
                      resolve_unosat_context, append_scene_log)

print("Label folder: ", LABELED_FOOTPRINT_DIR)
print("Raster folder:", RASTER_DIR)
print("Patch folder: ", PROCESSED_DIR)
print(f"\nPatch tag: {prep_tag()}")
print("Preprocessing settings:")
for k, v in PREP.items():
    print(f"  {k}: {v}")

print(f"\nTemporal offsets (days): {temporal_offsets()}")
print(f"Development cities: {development_cities()}")
print(f"Holdout cities:     {held_out_cities()}")

print("\nRegistered cities and dates:")
for city, info in CITY_REGISTRY.items():
    on_disk = list_label_dates(city)
    missing = [d for d in info["label_dates"] if d not in on_disk]
    extra = [d for d in on_disk if d not in info["label_dates"]]
    print(f"  {city} [{city_role(city)}]: {len(info['label_dates'])} dates, "
          f"war start {info['war_start']}")
    print(f"     using: {info['label_dates']}")
    if missing:
        print(f"     missing label files in {LABELED_FOOTPRINT_DIR}: {missing}")
    if extra:
        # CITY_REGISTRY determines which dates enter preprocessing.
        print(f"     also on disk, not registered: {extra}")

    # Report invalid temporal configurations before export.
    for date, reason in check_city_windows(city):
        print(f"     unusable {f'date {date}' if date else 'city'}: {reason}")

    jobs = post_jobs(city)
    lab = [d for d, w, is_lab in jobs if is_lab]
    off = [d for d, w, is_lab in jobs if not is_lab]
    print(f"     post composites needed: {len(jobs)} "
          f"({len(lab)} labelled + {len(off)} unlabelled offsets)")
    print(f"     offsets: {off}")

### 1b. Existing outputs

Export and patch-cutting functions skip complete outputs. The inventory below
shows which rasters and patch arrays already exist and which operations remain.

`to_export` and `to_cut` count the remaining Earth Engine exports and local
patch-cutting operations.


In [ ]:
# Summarize complete and missing preprocessing outputs.
overview = inventory_summary()
print(overview.to_string(index=False))

todo = overview[(overview["to_export"] > 0) | (overview["to_cut"] > 0)]
done = overview[(overview["to_export"] == 0) & (overview["to_cut"] == 0)]

print(f"\nComplete cities: "
      f"{list(done['city']) if len(done) else 'none'}")
if len(todo):
    print(f"Remaining work: {overview['to_export'].sum()} composites to export, "
          f"{overview['to_cut'].sum()} patch arrays to cut, for "
          f"{list(todo['city'])}")
    # Patch-array size scales with the number of buildings.
    print("\nMissing outputs by city:")
    for city in todo["city"]:
        inv = inventory(city)
        miss = inv[~inv["raster"] | ~inv["patches"]]
        print(f"\n{city} [{city_role(city)}]")
        print(miss[["kind", "date", "sensor", "raster", "patches"]]
              .to_string(index=False))
else:
    print("All registered cities are fully preprocessed.")


## 2. Earth Engine export

**One pre-war composite per city.** The fixed window ends at `war_start` and
spans `PREP["imagery"]["pre_months"]`. It is reused by every labelled and
temporal-offset date.

**One post composite per date**, including the unlabelled offset dates, is
built from the `n` acquisition days closest in time, searched backward and
forward. Same-day slices are mosaicked first. The configured fallback dates
fill only native nodata. When at least one fallback is configured, the
unfilled primary is also exported for the diagnostic in Section 4.

**The relative orbit is pinned per city.** Mixing orbits injects viewing-angle
differences that look like change but are not damage. The selected orbit and
bounding box are written once per city to
`rasters/{city}_export.json`, along with the bounding box, so every raster
covers the same ground on the same grid. Orbit selection is a coverage check;
the nearest-scene rule separately chooses dates on that orbit.

The orbit is chosen by *measured coverage of the city* across the pre window
and all labelled post windows, not by image count, since a scene clipping one
corner counts the same as one covering it fully.

Exports are written directly to Drive. Existing complete files are reused.


In [ ]:
# S1_EARLIEST provides one availability cutoff for window checks and orbit selection.


def _init_ee():
    project = PREP["imagery"]["gee_project"]
    try:
        ee.Initialize(project=project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=project)
    return ee


def _iso(date):
    """'20240503' -> '2024-05-03'."""
    return f"{date[:4]}-{date[4:6]}-{date[6:8]}"


def pre_window(city):
    """Return the fixed pre-war S1 baseline window for a city."""
    im = PREP["imagery"]
    war = pd.Timestamp(CITY_REGISTRY[city]["war_start"])
    return ((war - pd.DateOffset(months=im["pre_months"])).strftime("%Y-%m-%d"),
            war.strftime("%Y-%m-%d"))


def city_bbox(city):
    """Export area, decided ONCE per city and cached.

    Every raster of a city must cover the same ground: if the pre raster and
    a post raster had different extents, a building could sit comfortably
    inside one and off the edge of the other, and the two patch arrays would
    disagree about which rows are usable.
    """
    meta = read_export_meta(city)
    if "bbox" in meta:
        return meta["bbox"]
    bounds = []
    for date in CITY_REGISTRY[city]["label_dates"]:
        bounds.append(load_labels(city, date).total_bounds)
    b = np.array(bounds)
    pad = PREP["imagery"]["aoi_buffer_deg"]
    bbox = [float(b[:, 0].min() - pad), float(b[:, 1].min() - pad),
            float(b[:, 2].max() + pad), float(b[:, 3].max() + pad)]
    write_export_meta(city, bbox=bbox)
    print(f"  {city} bbox: {[round(v, 4) for v in bbox]}")
    return bbox

In [ ]:
def _s1_collection(ee, aoi):
    """Sentinel-1 IW, VV+VH, preserving GEE's native validity mask.

    Do not threshold low dB values: a finite value below -35 dB can be a
    genuine weak return. GEE's own nodata mask is retained, and only those
    genuinely missing pixels are eligible for temporal fallback.
    """
    return (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(aoi)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
            .select(["VV", "VH"]))


def _scene_date(scene_id):
    """'S1A_IW_GRDH_1SDV_20240410T034432_...' -> '2024-04-10'."""
    raw = scene_id.split("_")[4][:8]
    return f"{raw[:4]}-{raw[4:6]}-{raw[6:8]}"


def nearest_s1_post(ee, aoi, orbit, date, n, search_days):
    """Return the n distinct S1 acquisition days closest to `date`.

    All same-day slices on the pinned orbit are mosaicked before a day is
    counted. Thus n=1 means one temporal observation, not one spatial GEE
    asset. Each daily mosaic carries signed_days, abs_days, acquisition_day,
    scene_ids and slice_count properties and uses one shared VV/VH mask.

    Searching on both sides of the target limits temporal distance from the
    assessment. The optional fallback observations are selected from the same
    ordered collection.
    """
    target = ee.Date(_iso(date))
    a = target.advance(-search_days, "day")
    b = target.advance(search_days, "day")

    def tag_distance(img):
        signed = img.date().difference(target, "day")
        return img.set("signed_days", signed, "abs_days", signed.abs(),
                       "acquisition_day", img.date().format("YYYY-MM-dd"))

    tagged = (_s1_collection(ee, aoi).filterDate(a, b)
              .filter(ee.Filter.eq("relativeOrbitNumber_start", orbit))
              .map(tag_distance))
    representatives = (tagged.distinct(["acquisition_day"])
                       .sort("abs_days").limit(n))

    def mosaic_day(rep):
        rep = ee.Image(rep)
        day = ee.Date.parse("YYYY-MM-dd",
                            ee.String(rep.get("acquisition_day")))
        slices = tagged.filterDate(day, day.advance(1, "day"))
        image = slices.mosaic().select(["VV", "VH"])
        # Keep VV and VH from the same acquisition at every pixel.
        shared_mask = image.mask().reduce(ee.Reducer.min())
        return (image.updateMask(shared_mask)
                .copyProperties(rep, ["signed_days", "abs_days",
                                      "acquisition_day"])
                .set("scene_ids", slices.aggregate_array("system:index"),
                     "slice_count", slices.size()))

    daily = representatives.toList(n).map(mosaic_day)
    return ee.ImageCollection.fromImages(daily).sort("abs_days")


def choose_orbit(city, force=False):
    """Pick one relative orbit for the whole city and remember it.

    Scored by the WORST coverage of the pre-period union and each labelled
    date's actual closest acquisition-day mosaic. The coverage check thus
    uses exactly the same spatial/temporal unit as export_s1_post().
    """
    meta = read_export_meta(city)
    orbit_method = "nearest_day_native_v1"
    if ("orbit" in meta and meta.get("orbit_method") == orbit_method
            and not force):
        return meta["orbit"]

    ee = _init_ee()
    aoi = ee.Geometry.Rectangle(city_bbox(city))
    s1 = _s1_collection(ee, aoi)

    windows = {"pre": pre_window(city)}
    search_days = PREP["imagery"]["post_search_days"]
    for date in CITY_REGISTRY[city]["label_dates"]:
        target = pd.Timestamp(_iso(date))
        windows[date] = ((target - pd.Timedelta(days=search_days)).strftime("%Y-%m-%d"),
                         (target + pd.Timedelta(days=search_days)).strftime("%Y-%m-%d"))

    if windows["pre"][0] < S1_EARLIEST:
        print(f"  Pre window begins before the S1 availability date "
              f"({S1_EARLIEST}).")

    counts = {}
    for name, (a, b) in windows.items():
        h = (s1.filterDate(a, b)
               .aggregate_histogram("relativeOrbitNumber_start").getInfo())
        counts[name] = {int(float(k)): int(v) for k, v in h.items()}

    shared = set.intersection(*[set(c) for c in counts.values()])
    if not shared:
        raise RuntimeError(f"{city}: no single orbit covers every window. {counts}")

    def coverage(name, a, b, orbit):
        if name == "pre":
            c = s1.filterDate(a, b).filter(
                ee.Filter.eq("relativeOrbitNumber_start", orbit))
        else:
            c = nearest_s1_post(ee, aoi, orbit, name, 1, search_days)
        m = c.select("VV").count().gt(0).unmask(0)
        return m.reduceRegion(ee.Reducer.mean(), aoi, 200, maxPixels=1e9).get("VV")

    cov = ee.Dictionary({
        f"{o}|{name}": coverage(name, a, b, o)
        for o in shared for name, (a, b) in windows.items()}).getInfo()

    scored = {o: min((cov[f"{o}|{n}"] or 0) for n in windows) for o in shared}
    best = max(scored, key=lambda o: (scored[o], min(counts[n][o] for n in windows)))
    for o in sorted(shared):
        per = "  ".join(f"{n}:{counts[n][o]:2d}/{cov[f'{o}|{n}'] or 0:.2f}"
                        for n in windows)
        print(f"  orbit {o}: {per}")
    print(f"  {city}: using orbit {best}, worst-case coverage {scored[best]:.2f}")
    if scored[best] < PREP["imagery"]["min_aoi_coverage"]:
        print(f"  Best-orbit coverage is {scored[best]*100:.0f} percent in "
              f"the least-covered window; remaining pixels are nodata.")
    write_export_meta(city, orbit=int(best), orbit_coverage=float(scored[best]),
                      orbit_method=orbit_method)
    return int(best)


In [ ]:
def _download(ee, image, aoi, out):
    tmp = out + ".part"
    geemap.download_ee_image(image, tmp, region=aoi,
                             scale=PREP["imagery"]["scale"], crs="EPSG:4326")
    os.replace(tmp, out)          # rename last, so a killed cell leaves no
    print(f"  exported {os.path.basename(out)}")   # half-file that looks done


def export_s1_pre(city):
    out = pre_raster_path(city, "s1")
    if os.path.exists(out):
        print(f"  {os.path.basename(out)} exists, skipping")
        return out
    ee = _init_ee()
    aoi = ee.Geometry.Rectangle(city_bbox(city))
    orbit = choose_orbit(city)
    a, b = pre_window(city)
    col = (_s1_collection(ee, aoi).filterDate(a, b)
           .filter(ee.Filter.eq("relativeOrbitNumber_start", orbit)))
    n = col.size().getInfo()
    print(f"  {city} pre-war {a} to {b}: {n} scenes on orbit {orbit}")
    img = col.median().rename(["pre_VV", "pre_VH"]).clip(aoi).toFloat()
    _download(ee, img, aoi, out)
    return out


def export_s1_post(city, date, window_days=None):
    """Export the closest acquisition plus native-nodata fallback.

    n controls the primary temporal composite (normally one acquisition
    day). PREP['imagery']['post_fallback_dates'] additional distinct days
    are tried in nearest-first order ONLY where the primary is natively
    masked. Valid primary pixels are never averaged or replaced. When at
    least one fallback date is configured, the raw primary is also exported
    for the fallback diagnostic.
    """
    n = window_days if window_days else PREP["imagery"]["post_n_labelled"]
    fallback_n = int(PREP["imagery"].get("post_fallback_dates", 0))
    needed = n + fallback_n
    out = post_raster_path(city, date, "s1", n)
    primary_out = post_primary_raster_path(city, date, "s1", n)
    need_primary = fallback_n > 0
    if os.path.exists(out) and (not need_primary or os.path.exists(primary_out)):
        print(f"  {os.path.basename(out)} exists, skipping")
        return out

    ee = _init_ee()
    aoi = ee.Geometry.Rectangle(city_bbox(city))
    orbit = choose_orbit(city)
    search_days = PREP["imagery"]["post_search_days"]
    col = nearest_s1_post(ee, aoi, orbit, date, needed, search_days)

    days = col.aggregate_array("acquisition_day").getInfo()
    signed = col.aggregate_array("signed_days").getInfo()
    scene_ids = col.aggregate_array("scene_ids").getInfo()
    if len(days) < needed:
        raise RuntimeError(
            f"{city} {date}: only found {len(days)}/{needed} distinct "
            f"acquisition dates within +/-{search_days} days on orbit "
            f"{orbit}; primary n={n}, fallback dates={fallback_n}.")

    daily = col.toList(needed)
    primary_col = ee.ImageCollection.fromImages(daily.slice(0, n))
    primary = primary_col.median().select(["VV", "VH"])
    primary = primary.updateMask(primary.mask().reduce(ee.Reducer.min()))
    filled = primary
    for rank in range(1, fallback_n + 1):
        fallback = ee.Image(daily.get(n + rank - 1)).select(["VV", "VH"])
        filled = filled.unmask(fallback, False)

    unosat_date, offset_days = resolve_unosat_context(city, date)
    kind = "labelled" if offset_days == 0 else f"offset {offset_days:+d}d"
    print(f"  {city} {date} post ({kind}); primary n={n}, "
          f"fallback dates={fallback_n}:")
    log_rows = []
    for rank, (day, sd, ids) in enumerate(zip(days, signed, scene_ids)):
        role = "primary" if rank < n else f"fallback_{rank - n + 1}"
        days_from_unosat = sd + offset_days
        tail = ("" if offset_days == 0 else
                f", {days_from_unosat:+.1f}d from UNOSAT date {unosat_date}")
        print(f"    {role:10s} {day} ({len(ids)} slice(s)), "
              f"{sd:+.1f}d from target{tail}")
        for sid in ids:
            log_rows.append({
                "city": city,
                "kind": "labelled" if offset_days == 0 else "offset",
                "unosat_date": unosat_date, "offset_days": offset_days,
                "target_date": date,
                "post_variant": f"nearest{n}_fb{fallback_n}_native",
                "source_role": role,
                "acquisition_day": day, "scene_id": sid,
                "scene_date": _scene_date(sid),
                "days_from_target": sd,
                "days_from_unosat": days_from_unosat,
            })

    filled_img = filled.rename(["post_VV", "post_VH"]).clip(aoi).toFloat()
    if need_primary and not os.path.exists(primary_out):
        primary_img = primary.rename(["post_VV", "post_VH"]).clip(aoi).toFloat()
        _download(ee, primary_img, aoi, primary_out)
    if not os.path.exists(out):
        _download(ee, filled_img, aoi, out)
    append_scene_log(city, log_rows)
    return out

In [ ]:
# Export labelled dates before temporal offsets. Per-city failures are recorded
# without discarding completed files from other cities.

failures = {}
for city, info in CITY_REGISTRY.items():
    print(f"\n=== {city} [{city_role(city)}] ===")

    bad = [f"{d}: {why}" for d, why in check_city_windows(city) if d]
    if bad:
        print("  Skipped because assessment dates are unusable:")
        for b in bad:
            print(f"    {b}")
        failures[city] = "unusable windows"
        continue

    try:
        city_bbox(city)
        export_s1_pre(city)
        for date, window_days, is_labelled in post_jobs(city):
            kind = "labelled" if is_labelled else "offset"
            print(f"  -- {date} ({kind})")
            export_s1_post(city, date, window_days)
    except Exception as e:
        print(f"  Failed: {type(e).__name__}: {e}")
        failures[city] = f"{type(e).__name__}: {e}"

print("\n" + "=" * 60)
if failures:
    print("Export failures:")
    for city, why in failures.items():
        print(f"  {city}: {why}")
else:
    print("All registered cities exported.")
print("\nCurrent inventory:")
print(inventory_summary().to_string(index=False))


### 2c. Speckle diagnostic for one versus two acquisition days

`post_n_labelled=1` retains a single acquisition day. Using two days can
reduce speckle but increases the mean temporal distance from the assessment.
This optional diagnostic uses Earth Engine to measure the difference over
Gaza. The later EDA sections use only the exported files.

**The method.** SAR speckle is per-pixel noise: in a truly uniform patch of
ground, any pixel-to-pixel variation is noise, not signal, so averaging more
looks should shrink it. This approximates that with a moving 7x7 pixel
(70x70 m) window. Local standard deviation is used as a noise proxy, and the
the **median** of that local std across the whole city (not the mean), so
genuine edges (real buildings, streets) don't dominate the number, since
only a minority of pixels sit right on one.

**The benchmark.** Two independent looks should cut the standard deviation
by `1/sqrt(2)`, a 29.3 percent reduction. A measured reduction well below
that indicates that the statistic also contains real urban texture, which is
not reduced by averaging additional looks.


In [ ]:
def s1_composite(city, date, n, search_days=None, band="VV"):
    """The n closest-in-time S1 acquisition days, as one image (median if
    n > 1). Same scene selection as nearest_s1_post(), for direct pixel
    access rather than a raster export."""
    ee = _init_ee()
    aoi = ee.Geometry.Rectangle(city_bbox(city))
    orbit = choose_orbit(city)
    search_days = search_days or PREP["imagery"]["post_search_days"]
    col = nearest_s1_post(ee, aoi, orbit, date, n, search_days)
    return col.select(band).median().clip(aoi)


def local_noise(image, aoi, kernel_radius=3, scale=10):
    """Median local standard deviation across an AOI - a speckle proxy.
    See the markdown above for why local windows + a median summary."""
    ee = _init_ee()
    kernel = ee.Kernel.square(radius=kernel_radius, units="pixels")
    local_std = image.reduceNeighborhood(ee.Reducer.stdDev(), kernel)
    stats = local_std.reduceRegion(ee.Reducer.median(), aoi, scale,
                                   maxPixels=1e9, bestEffort=True)
    return stats.getInfo()


def compare_noise(city, date, band="VV", kernel_radius=3):
    """n=1 vs n=2 local-noise comparison for one assessment date."""
    aoi = ee.Geometry.Rectangle(city_bbox(city))
    img1 = s1_composite(city, date, n=1, band=band)
    img2 = s1_composite(city, date, n=2, band=band)
    s1v = local_noise(img1, aoi, kernel_radius)[f"{band}_stdDev"]
    n2v = local_noise(img2, aoi, kernel_radius)[f"{band}_stdDev"]
    return {"city": city, "date": date, "n1_std_dB": s1v, "n2_std_dB": n2v,
           "measured_%": 100 * (1 - n2v / s1v), "theoretical_%": 29.3}


ee = _init_ee()
noise_rows = [compare_noise("Gaza", d) for d in CITY_REGISTRY["Gaza"]["label_dates"]]
noise_df = pd.DataFrame(noise_rows)
print(noise_df.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.4))
x = np.arange(len(noise_df))
w = 0.35
ax.bar(x - w / 2, noise_df["n1_std_dB"], w, label="n=1", color="C3")
ax.bar(x + w / 2, noise_df["n2_std_dB"], w, label="n=2", color="C0")
ax.set_xticks(x)
ax.set_xticklabels(noise_df["date"], rotation=45)
ax.set_ylabel("median local std (dB)")
ax.set_title("Gaza: patch-scale speckle proxy, n=1 vs n=2")
ax.legend()
plt.tight_layout()
plt.show()

mean_measured = noise_df["measured_%"].mean()
print(f"\nmean measured reduction: {mean_measured:.1f} percent "
      f"(theory for 2 independent looks: 29.3 percent)")

# Rough texture/speckle split. Model the local-std measurement as
# var = T + S, with T = constant real texture and S = speckle variance that
# halves under n=2. Solving for S/T from the measured STD reduction:
frac_var_reduction = 1 - (1 - mean_measured / 100) ** 2
if frac_var_reduction < 0.5:
    s_over_t = frac_var_reduction / (0.5 - frac_var_reduction)
    texture_share = 1 / (1 + s_over_t)
    print(f"implied split of the measured variance: "
          f"~{texture_share * 100:.0f} percent real texture, "
          f"~{(1 - texture_share) * 100:.0f} percent speckle")

print(f"Measured reduction relative to the 29.3 percent independent-look "
      f"benchmark: {mean_measured / 29.3 * 100:.1f} percent")


## 3. The canonical building table

One table per city, holding the buildings that exist at **every** assessment
date, sorted by `system:index` and sampled by id rather than by row position.
Every patch array is aligned to it positionally, so joining a pre patch to a
post patch, or a May score to a July score, needs no id matching at all.

Columns: `system:index`, geometry, `area`, coordinates (`lon`/`lat` and
`x_m`/`y_m` in metres for the neighbour searches), and one `class_{date}`
column per assessment. `severity_{date}` and `confidence_{date}` are retained
when they are present in the label files.

Holding the label matrix in one table is also what makes multi-date
validation cheap in notebook 2: the same building at three dates is three
rows referring to the same patch geometry and three different post arrays.


In [ ]:
def build_building_table(city, force=False):
    path = buildings_path(city)
    if os.path.exists(path) and not force:
        tab = gpd.read_parquet(path)
        print(f"{city}: table exists, {len(tab):,} buildings")
        return tab

    dates = CITY_REGISTRY[city]["label_dates"]
    frames = {}
    for d in dates:
        g = load_labels(city, d)
        if g["system:index"].duplicated().any():
            raise ValueError(f"{city} {d}: duplicate building identifiers")
        frames[d] = g.set_index("system:index")
        print(f"  {city} {d}: {len(frames[d]):,} rows, "
              f"{frames[d]['class'].mean()*100:.1f} percent damaged")

    id_sets = {d: set(f.index) for d, f in frames.items()}
    reference_ids = id_sets[dates[0]]
    for d in dates[1:]:
        missing = reference_ids - id_sets[d]
        extra = id_sets[d] - reference_ids
        if missing or extra:
            raise ValueError(
                f"{city} {d}: footprint rows differ from {dates[0]} "
                f"({len(missing)} missing, {len(extra)} extra)")
    ids = reference_ids
    print(f"  aligned across {len(dates)} dates: {len(ids):,} buildings")

    base = frames[dates[-1]].loc[sorted(ids)].reset_index()
    base = sample_buildings(base, PREP["n_sample"], PREP["seed"])
    base = base.sort_values("system:index").reset_index(drop=True)

    keep = ["system:index", "geometry"] + [c for c in ["area"] if c in base]
    # a GeoDataFrame, not a plain frame: parquet needs the geometry metadata
    # for notebook 3 to read the footprints back as polygons
    tab = gpd.GeoDataFrame(base[keep].copy(), geometry="geometry", crs="EPSG:4326")
    cen = base["centroid"]
    merc = cen.to_crs(3857)
    tab["lon"] = cen.apply(lambda p: p.x).to_numpy(float)
    tab["lat"] = cen.apply(lambda p: p.y).to_numpy(float)
    tab["x_m"] = merc.apply(lambda p: p.x).to_numpy(float)
    tab["y_m"] = merc.apply(lambda p: p.y).to_numpy(float)

    for d in dates:
        f = frames[d]
        tab[f"class_{d}"] = tab["system:index"].map(f["class"]).to_numpy(int)
        for extra in ["severity", "confidence"]:
            if extra in f.columns:
                tab[f"{extra}_{d}"] = tab["system:index"].map(f[extra]).to_numpy(float)

    tab.to_parquet(path)
    print(f"  saved {os.path.basename(path)}: {len(tab):,} buildings, "
          f"{len(dates)} label columns")
    return tab


# Record per-city failures without discarding tables already completed.
tables = {}
for city in CITY_REGISTRY:
    try:
        tables[city] = build_building_table(city)
    except Exception as e:
        print(f"{city}: failed: {type(e).__name__}: {e}")

print(f"\nBuilt tables for: {list(tables)}")
print(pd.DataFrame([
    {"city": c, "role": city_role(c), "buildings": len(t),
     "label_cols": sum(k.startswith("class_") for k in t.columns)}
    for c, t in tables.items()]).to_string(index=False))


## 4. Cut patches

One patch per building per raster, centred on its centroid. A row is written
for **every** building in the table whether or not its patch is usable, and a
separate boolean `valid` array records which rows actually hold pixels. That
is what keeps the arrays positionally aligned: a building that falls off the
edge of one date's raster becomes a masked row there rather than a missing
row that shifts everything after it.

Radar nodata can be negative infinity in dB rather than NaN, so `np.isfinite`
is used instead of `np.isnan`.

Patches are written straight into a memory-mapped `.npy` in chunks, so the
full stack never exists in RAM. They are stored as float16: backscatter in dB
spans about -35 to +5, where float16 resolves about 0.01 dB, far finer than radar
speckle. Uncompressed and in their own file, so notebook 2 can memory-map
them. A compressed `.npz` member has no fixed position on disk and would
have to be decompressed whole.


In [ ]:
CHUNK = 4096   # rows buffered before writing to the memory map


def _corners(paths, tab, patch):
    """Per-raster top-left patch corners, plus which buildings fit."""
    half = patch // 2
    lons, lats = tab["lon"].to_numpy(float), tab["lat"].to_numpy(float)
    out = []
    for p in paths:
        with rasterio.open(p) as src:
            cols, rows = (~src.transform) * (lons, lats)
            r0 = np.round(rows).astype(int) - half
            c0 = np.round(cols).astype(int) - half
            inside = ((r0 >= 0) & (c0 >= 0) &
                      (r0 + patch <= src.height) & (c0 + patch <= src.width))
            out.append((p, r0, c0, inside, src.count))
    return out


def cut_phase(city, tab, phase, date=None, window_days=None, force=False):
    """Write the pre (or one post) patch array plus its valid mask."""
    patch = PREP["patch_size"]
    if phase == "pre":
        paths = [pre_raster_path(city, "s1")]
        x_path, v_path = pre_patch_paths(city)
    else:
        paths = [post_raster_path(city, date, "s1", window_days)]
        x_path, v_path = post_patch_paths(city, date, window_days)

    label = f"{city} {phase}" + (f" {date}" if date else "")
    if os.path.exists(x_path) and os.path.exists(v_path) and not force:
        print(f"  {label}: already cut")
        return
    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing source raster: {p}")

    info = _corners(paths, tab, patch)
    n = len(tab)
    n_ch = sum(c for *_, c in info)
    fits = np.logical_and.reduce([i[3] for i in info])

    arrays = []
    for p, *_ in info:
        with rasterio.open(p) as src:
            arrays.append(src.read().astype(np.float32))

    tmp = x_path + ".part"
    out = open_patch_writer(tmp, n, n_ch, patch)
    valid = np.zeros(n, bool)
    buf = np.zeros((CHUNK, n_ch, patch, patch), np.float16)
    buf_idx = []

    def flush():
        if buf_idx:
            out[np.array(buf_idx)] = buf[:len(buf_idx)]
            buf_idx.clear()

    for j in np.where(fits)[0]:
        parts, good = [], True
        for arr, (_, r0, c0, _, _) in zip(arrays, info):
            block = arr[:, r0[j]:r0[j] + patch, c0[j]:c0[j] + patch]
            if not np.isfinite(block).all():
                good = False
                break
            parts.append(block)
        if not good:
            continue
        buf[len(buf_idx)] = np.concatenate(parts, axis=0).astype(np.float16)
        buf_idx.append(j)
        valid[j] = True
        if len(buf_idx) == CHUNK:
            flush()
    flush()

    out.flush()
    del out, arrays
    os.replace(tmp, x_path)
    np.save(v_path, valid)
    print(f"  {label}: {valid.sum():,}/{n:,} usable patches, {n_ch} channels, "
          f"{os.path.getsize(x_path)/1e9:.2f} GB")


# Record per-city failures without discarding completed patch arrays.
cut_failures = {}
for city, tab in tables.items():
    print(f"\n=== {city} [{city_role(city)}] ===")
    try:
        cut_phase(city, tab, "pre")
        for date, window_days, is_labelled in post_jobs(city):
            cut_phase(city, tab, "post", date, window_days)
    except Exception as e:
        print(f"  Failed: {type(e).__name__}: {e}")
        cut_failures[city] = f"{type(e).__name__}: {e}"

print("\n" + "=" * 60)
if cut_failures:
    print("Patch-cutting failures:")
    for city, why in cut_failures.items():
        print(f"  {city}: {why}")
else:
    print("Every city with a building table has complete patch arrays.")
print("\nCurrent inventory:")
print(inventory_summary().to_string(index=False))


In [ ]:
# How many buildings survive at each date, and at each temporal offset.
#
# The labelled dates are what the CNN trains and is evaluated on, so losses
# there cost data directly. Losses at an offset date are cheaper: notebook 2
# fills those scores with NaN and lets XGBoost route around the gap, so an
# offset with slightly fewer buildings is a missing feature value, not a
# dropped row.
for city in CITY_REGISTRY:
    try:
        d = load_city(city)
    except FileNotFoundError as e:
        print(f"\n{city}: preprocessing incomplete: {e}")
        continue

    n = len(d["table"])
    print(f"\n{city} [{city_role(city)}]: {n:,} buildings in the table, "
          f"{d['pre_valid'].sum():,} with a usable pre patch")
    rows = []
    for date, window_days, is_labelled in post_jobs(city):
        try:
            ok = usable_mask(d, date, window_days)
        except FileNotFoundError:
            rows.append({"date": date,
                         "kind": "labelled" if is_labelled else "offset",
                         "usable": 0, "percent": 0.0})
            continue
        rows.append({"date": date,
                     "kind": "labelled" if is_labelled else "offset",
                     "usable": int(ok.sum()),
                     "percent": round(100 * ok.mean(), 2)})
    df = pd.DataFrame(rows).sort_values("date")
    print(df.to_string(index=False))
    worst = df["percent"].min()
    if worst < 90:
        print(f"   Lowest usable share: {worst:.0f} percent of buildings.")


In [ ]:
def raster_patch_valid(path, tab, patch=None):
    """Return patch and pixel validity using the same rule as cut_phase()."""
    patch = patch or PREP["patch_size"]
    p, r0, c0, inside, _ = _corners([path], tab, patch)[0]
    with rasterio.open(p) as src:
        values = src.read().astype(np.float32)
    pixel_ok = np.isfinite(values).all(axis=0)
    valid = np.zeros(len(tab), bool)
    bad = (~pixel_ok).astype(np.uint8)
    summed = np.pad(bad.cumsum(0).cumsum(1), ((1, 0), (1, 0)))
    j = np.where(inside)[0]
    bad_count = (summed[r0[j] + patch, c0[j] + patch]
                 - summed[r0[j], c0[j] + patch]
                 - summed[r0[j] + patch, c0[j]]
                 + summed[r0[j], c0[j]])
    valid[j] = bad_count == 0
    return valid, pixel_ok


def fallback_diagnostic(city="Gaza", bins=80):
    fallback_n = int(PREP["imagery"].get("post_fallback_dates", 0))
    if fallback_n == 0:
        print("Fallback diagnostic: disabled because post_fallback_dates=0.")
        return pd.DataFrame()

    d = load_city(city)
    tab = d["table"]
    lat, lon = d["lat"], d["lon"]
    pre_ok = d["pre_valid"]
    dates = CITY_REGISTRY[city]["label_dates"]
    print(f"Pre-war usable: {pre_ok.sum():,}/{len(pre_ok):,} "
          f"({pre_ok.mean() * 100:.2f}%)")

    fig, axes = plt.subplots(2, len(dates),
                             figsize=(4.6 * len(dates), 9.0), squeeze=False)
    rows = []
    for col_i, dt in enumerate(dates):
        primary_path = post_primary_raster_path(city, dt, "s1")
        filled_path = post_raster_path(city, dt, "s1")
        if not os.path.exists(primary_path) or not os.path.exists(filled_path):
            raise FileNotFoundError(
                f"{dt}: missing primary or filled fallback diagnostic raster")

        primary_ok, primary_pixels = raster_patch_valid(primary_path, tab)
        filled_ok, filled_pixels = raster_patch_valid(filled_path, tab)
        if (primary_pixels & ~filled_pixels).any():
            raise RuntimeError(f"{dt}: fallback lost valid primary pixels")
        stored_ok = post_arrays(d, dt)[1]
        if not np.array_equal(filled_ok, stored_ok):
            raise RuntimeError(f"{dt}: stored patch mask differs from filled raster")

        missing_before = pre_ok & ~primary_ok
        missing_after = pre_ok & ~stored_ok
        resolved = missing_before & stored_ok
        if (primary_ok & ~stored_ok).any():
            raise RuntimeError(f"{dt}: fallback lost valid primary patches")

        rows.append({
            "date": dt,
            "primary missing pixels": int((~primary_pixels).sum()),
            "pixels resolved by fallback": int((~primary_pixels & filled_pixels).sum()),
            "still missing pixels": int((~filled_pixels).sum()),
            "primary missing patches": int(missing_before.sum()),
            "resolved by fallback": int(resolved.sum()),
            "still missing patches": int(missing_after.sum()),
        })

        total_h, xedges, yedges = np.histogram2d(lon, lat, bins=bins)
        stages = [(missing_before, "primary only"),
                  (missing_after, f"after {fallback_n} fallback dates")]
        for row_i, (mask, stage) in enumerate(stages):
            miss_h, _, _ = np.histogram2d(lon[mask], lat[mask],
                                            bins=[xedges, yedges])
            with np.errstate(invalid="ignore", divide="ignore"):
                fraction = np.where(total_h > 0, miss_h / total_h, np.nan)
            ax = axes[row_i, col_i]
            image = ax.imshow(
                fraction.T, origin="lower",
                extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
                cmap="Reds", vmin=0, vmax=1, aspect="auto")
            n_missing = int(mask.sum())
            ax.set_title(f"{dt}: {stage}\n{n_missing:,}/{len(mask):,} missing "
                         f"({100 * n_missing / len(mask):.1f}%)", fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])

    fig.colorbar(image, ax=list(axes.ravel()), shrink=0.7,
                 label="fraction of buildings missing a pixel per cell")
    plt.suptitle(
        f"{city}: native nodata before and after {fallback_n} fallback dates",
        y=1.01)
    plt.show()
    return pd.DataFrame(rows)


fallback_summary = fallback_diagnostic()
if not fallback_summary.empty:
    print("\nFallback resolution summary:")
    print(fallback_summary.to_string(index=False))


## 5. Gaza EDA

Everything below reads the processed files, so this section runs without an
Earth Engine session.

Gaza is the development city, so label-aware exploration and preprocessing
diagnostics are concentrated here. Section 6 reports the same descriptive
quantities for the registered holdout cities without fitting a model.


In [ ]:
CITY = "Gaza"
dates = CITY_REGISTRY[CITY]["label_dates"]

summary = summarize_processed(CITY)
print(summary.to_string(index=False))

d = load_city(CITY)
print(f"\nPre channels:  {d['pre_channel_names']}")
print(f"Post channels: {d['post_channel_names']}")
print(f"Pre array:  {d['pre'].shape}, {d['pre'].dtype}, "
      f"{d['pre'].nbytes/1e9:.2f} GB on disk (memory mapped, not loaded)")
post_last, _ = post_arrays(d, dates[-1])
print(f"Post array: {post_last.shape} per date, "
      f"{post_last.nbytes/1e9:.2f} GB each")
print(f"\nStoring pre once instead of per date saves "
      f"{d['pre'].nbytes * (len(post_jobs(CITY)) - 1) / 1e9:.1f} GB")


In [ ]:
# Damage prevalence and newly damaged buildings by assessment date.
x = pd.to_datetime(dates)
new = np.diff(summary["damaged"].to_numpy(), prepend=0)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(x, summary["damaged %"], marker="o", color="C3")
ax[0].set_ylabel("percent damaged")
ax[0].set_title(f"{CITY}: cumulative damage")
ax[1].bar(x, new, width=18, color="C0")
ax[1].set_ylabel("buildings")
ax[1].set_title("newly labelled damaged since previous assessment")
for a in ax:
    a.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"Damaged share ranges from {summary['damaged %'].min():.2f} to "
      f"{summary['damaged %'].max():.2f} percent across the registered dates.")


In [ ]:
# Spatial distribution of labels at each assessment date. Only coordinates
# and labels are read in this diagnostic.
tab = d["table"]
fig, axes = plt.subplots(1, len(dates), figsize=(4 * len(dates), 6))
axes = np.atleast_1d(axes)
for ax, dt in zip(axes, dates):
    y = tab[f"class_{dt}"].to_numpy(int)
    lat, lon = d["lat"], d["lon"]
    ax.scatter(lon[y == 0], lat[y == 0], s=0.05, alpha=0.15, color="C0", label="intact")
    ax.scatter(lon[y == 1], lat[y == 1], s=0.05, alpha=0.15, color="C3", label="damaged")
    ax.set_title(f"{dt}  ({y.mean() * 100:.1f} percent)")
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(markerscale=6, loc="lower left")
plt.tight_layout()
plt.show()


In [ ]:
# Compare patch-mean backscatter before and after the event. The difference
# is measured relative to each building's pre-war baseline in decibels.
last = dates[-1]
ok = usable_mask(d, last)
y = tab[f"class_{last}"].to_numpy(int)[ok]

pi = d["pre_channel_names"].index("s1_pre_VV")
qi = d["post_channel_names"].index("s1_post_VV")
post_arr, _ = post_arrays(d, last)

# Reading one channel of a memory map loads only that channel from disk.
pre_vv = d["pre"][ok, pi].mean(axis=(1, 2)).astype(np.float32)
post_vv = post_arr[ok, qi].mean(axis=(1, 2)).astype(np.float32)
diff = post_vv - pre_vv

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, (v, name) in zip(ax, [(pre_vv, "pre-war VV"), (post_vv, "post VV"),
                             (diff, "post minus pre VV")]):
    a.hist(v[y == 0], bins=60, alpha=0.6, density=True, label="intact")
    a.hist(v[y == 1], bins=60, alpha=0.6, density=True, label="damaged")
    a.set_xlabel("dB")
    a.set_title(name)
ax[2].axvline(0, color="k", ls="--", lw=1)
ax[0].legend()
plt.suptitle(f"{CITY} {last}: patch-mean backscatter", y=1.04)
plt.tight_layout()
plt.show()

print(f"mean post-minus-pre, intact:  {diff[y == 0].mean():+.2f} dB")
print(f"mean post-minus-pre, damaged: {diff[y == 1].mean():+.2f} dB")
print("Patch means are descriptive summaries; the CNN receives the complete patch.")


### Temporal signal diagnostic

This diagnostic computes the patch-mean ΔVV statistic across the offset dates.
It does not fit a model.

The intended temporal pattern is:

* separation between the classes increases or persists across offsets;
* the intact curve remains comparatively stable.

Flat, parallel curves indicate that the offsets add little information beyond
the labelled date.


In [ ]:
def temporal_signal(city, date, channel="VV"):
    """Patch-mean post-minus-pre at every offset, split by the date's label."""
    d = load_city(city)
    tab = d["table"]
    pi = d["pre_channel_names"].index(f"s1_pre_{channel}")
    qi = d["post_channel_names"].index(f"s1_post_{channel}")
    win = temporal_scene_count()

    # The pre-war baseline is the same at every offset, so read it once.
    pre_mean = np.full(len(tab), np.nan, np.float32)
    pv = d["pre_valid"]
    pre_mean[pv] = d["pre"][pv, pi].mean(axis=(1, 2)).astype(np.float32)

    rows = []
    for off, od in offset_dates(date).items():
        wd = None if off == 0 else win
        try:
            arr, _ = post_arrays(d, od, wd)
        except FileNotFoundError:
            print(f"  {od} not cut, skipping")
            continue
        ok = usable_mask(d, od, wd)
        y = tab[f"class_{date}"].to_numpy(int)[ok]
        dv = (arr[ok, qi].mean(axis=(1, 2)).astype(np.float32) - pre_mean[ok])
        rows.append({"offset": off, "date": od, "n": int(ok.sum()),
                     "intact": float(dv[y == 0].mean()),
                     "damaged": float(dv[y == 1].mean())})
    df = pd.DataFrame(rows).sort_values("offset")
    df["gap"] = df["intact"] - df["damaged"]
    return df


sig = temporal_signal(CITY, dates[0])
print(sig.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(sig["offset"], sig["intact"], marker="o", label="intact")
ax[0].plot(sig["offset"], sig["damaged"], marker="o", label="damaged")
ax[0].axvline(0, color="k", ls="--", lw=1)
ax[0].set_xlabel("days from the assessment date")
ax[0].set_ylabel("mean post minus pre VV (dB)")
ax[0].set_title(f"{CITY} {dates[0]}: signal across the offsets")
ax[0].legend()
ax[1].plot(sig["offset"], sig["gap"], marker="s", color="C3")
ax[1].axvline(0, color="k", ls="--", lw=1)
ax[1].set_xlabel("days from the assessment date")
ax[1].set_title("separation between the classes")
plt.tight_layout()
plt.show()

trend = sig["gap"].iloc[-1] - sig["gap"].iloc[0]
print(f"\nClass separation change from first to last offset: {trend:+.3f} dB")


In [ ]:
# Example patches: post-event VV for damaged and intact buildings, on a
# shared colour scale so they are comparable.
rng = np.random.default_rng(0)
idx = np.where(ok)[0]
dmg = rng.choice(idx[y == 1], 5, replace=False)
intact = rng.choice(idx[y == 0], 5, replace=False)
sample = np.asarray(post_arr[:5000, qi]).astype(np.float32)
vmin, vmax = np.percentile(sample, [2, 98])

fig, axes = plt.subplots(2, 5, figsize=(12, 5.2))
for row, (sel, name) in enumerate([(dmg, "damaged"), (intact, "intact")]):
    for col, j in enumerate(sel):
        axes[row, col].imshow(np.asarray(post_arr[j, qi]).astype(np.float32),
                              cmap="gray", vmin=vmin, vmax=vmax)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
    axes[row, 0].set_ylabel(name)
fig.suptitle(f"{CITY} {last}: post-event VV, "
             f"{PREP['patch_size']} px = {PREP['patch_size'] * 10} m across")
plt.tight_layout()
plt.show()


In [ ]:
# Label quality, using the extra columns the shared notebook 0 carried through.
g = d["table"]
sev = f"severity_{last}"
if sev in g.columns and g[sev].notna().any():
    print("UNOSAT severity of damaged buildings (1 = most severe):")
    print(g[sev].value_counts().sort_index().to_string())

# Compare damage prevalence across building-size quintiles. A strong trend
# indicates that footprint size may carry label information.
if "area" in g.columns:
    by_size = (g.groupby(pd.qcut(g["area"], 5, duplicates="drop"), observed=True)
                 [f"class_{last}"].agg(["mean", "size"]))
    by_size["mean"] = (by_size["mean"] * 100).round(1)
    print("\ndamaged percent by building-size quintile:")
    print(by_size.rename(columns={"mean": "damaged %", "size": "n"}).to_string())


## 6. The holdout cities

Raqqa, Mosul, Chernihiv and Rubizhne have `role: "holdout"`. They are
excluded from every part of notebook 2's Gaza split. Gaza's `test` band
therefore measures performance on unseen ground within the development city,
while these cities measure geographic transfer. No model is fitted here.

These summaries expose holdout labels. Model or threshold choices based on
the reported values would weaken the independence of the final evaluation.

The comparison reports four descriptive checks:

1. Class separation in patch-mean ΔVV.
2. Damage prevalence, which affects threshold-dependent metrics.
3. Building-size distributions and their relation to labels.
4. Data integrity, including usable-patch counts and label variation.


In [ ]:
# One row per city and date with sample size, prevalence, building area and
# patch-mean ΔVV separation. The same calculation is used for every city.


def patch_mean_delta(d, date, channel="VV"):
    """(delta VV per building, label, usable mask) for one city and date.

    Post minus the city's own pre-war baseline, so each city is compared
    against itself rather than against an absolute backscatter level.
    """
    pi = d["pre_channel_names"].index(f"s1_pre_{channel}")
    qi = d["post_channel_names"].index(f"s1_post_{channel}")
    arr, _ = post_arrays(d, date)
    ok = usable_mask(d, date)
    pre = d["pre"][ok, pi].mean(axis=(1, 2)).astype(np.float32)
    post = arr[ok, qi].mean(axis=(1, 2)).astype(np.float32)
    y = d["table"][f"class_{date}"].to_numpy(int)[ok]
    return post - pre, y, ok


def city_profile(city):
    """Comparable summary rows for every labelled date of one city."""
    d = load_city(city)
    tab = d["table"]
    rows = []
    for date in CITY_REGISTRY[city]["label_dates"]:
        diff, y, ok = patch_mean_delta(d, date)
        area = tab.loc[ok, "area"] if "area" in tab.columns else pd.Series(dtype=float)
        # Cohen's d expresses class separation relative to within-class spread.
        s = np.sqrt(((y == 1).sum() * diff[y == 1].var()
                     + (y == 0).sum() * diff[y == 0].var())
                    / max(len(y), 1)) if 0 < y.sum() < len(y) else np.nan
        rows.append({
            "city": city,
            "role": city_role(city),
            "date": date,
            "buildings": int(ok.sum()),
            "damaged %": round(100 * y.mean(), 2) if len(y) else np.nan,
            "median area m2": round(float(area.median()), 1) if len(area) else np.nan,
            "dVV intact": round(float(diff[y == 0].mean()), 3) if (y == 0).any() else np.nan,
            "dVV damaged": round(float(diff[y == 1].mean()), 3) if (y == 1).any() else np.nan,
            "separation dB": round(float(diff[y == 1].mean() - diff[y == 0].mean()), 3)
                              if 0 < y.sum() < len(y) else np.nan,
            "cohen d": round(float((diff[y == 1].mean() - diff[y == 0].mean()) / s), 3)
                        if s and np.isfinite(s) and s > 0 else np.nan,
        })
    return rows


profile_rows, unavailable = [], {}
for city in CITY_REGISTRY:
    try:
        profile_rows += city_profile(city)
    except (FileNotFoundError, KeyError) as e:
        unavailable[city] = f"{type(e).__name__}: {e}"

profiles = pd.DataFrame(profile_rows)
print(profiles.to_string(index=False))

if unavailable:
    print("\nnot preprocessed yet, so not shown above:")
    for city, why in unavailable.items():
        print(f"  {city}: {why}")

print("\n'separation dB' is damaged minus intact. 'cohen d' expresses the "
      "same difference relative to its within-class spread.")


In [ ]:
# Damage distribution in each city at its newest assessment. Each panel uses
# its own extent because the cities cover different areas.
shown = [c for c in CITY_REGISTRY if c not in unavailable]

fig, axes = plt.subplots(1, len(shown), figsize=(4.2 * len(shown), 4.6))
axes = np.atleast_1d(axes)
for ax, city in zip(axes, shown):
    dd = load_city(city)
    last_d = CITY_REGISTRY[city]["label_dates"][-1]
    yy = dd["table"][f"class_{last_d}"].to_numpy(int)
    ax.scatter(dd["lon"][yy == 0], dd["lat"][yy == 0], s=0.05, alpha=0.15,
               color="C0", label="intact")
    ax.scatter(dd["lon"][yy == 1], dd["lat"][yy == 1], s=0.05, alpha=0.15,
               color="C3", label="damaged")
    ax.set_title(f"{city} [{city_role(city)}]\n{last_d}  "
                 f"({yy.mean() * 100:.1f}% damaged, {len(yy):,} bldgs)",
                 fontsize=9)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(markerscale=6, loc="lower left", fontsize=7)
plt.suptitle("Damage footprint by city, newest assessment", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Compare the class-conditional ΔVV distributions across cities using the
# same statistic and axes.
fig, axes = plt.subplots(1, len(shown), figsize=(3.6 * len(shown), 3.2),
                         sharex=True)
axes = np.atleast_1d(axes)
for ax, city in zip(axes, shown):
    dd = load_city(city)
    last_d = CITY_REGISTRY[city]["label_dates"][-1]
    diff, yy, _ = patch_mean_delta(dd, last_d)
    ax.hist(diff[yy == 0], bins=60, alpha=0.6, density=True, label="intact")
    ax.hist(diff[yy == 1], bins=60, alpha=0.6, density=True, label="damaged")
    ax.axvline(0, color="k", ls="--", lw=1)
    sep = diff[yy == 1].mean() - diff[yy == 0].mean()
    ax.set_title(f"{city} {last_d}\nseparation {sep:+.2f} dB", fontsize=9)
    ax.set_xlabel("post minus pre VV (dB)")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=7)
plt.xlim(-8, 8)
plt.suptitle("Patch-mean ΔVV: does the Gaza signal exist elsewhere?", y=1.04)
plt.tight_layout()
plt.show()

# Building-size distributions provide context for fixed-size image patches.
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for city in shown:
    dd = load_city(city)
    if "area" not in dd["table"].columns:
        continue
    a = dd["table"]["area"].to_numpy(float)
    ax[0].hist(np.log10(a[a > 0]), bins=60, density=True, histtype="step",
               lw=1.6, label=f"{city} (med {np.median(a):.0f} m2)")
    last_d = CITY_REGISTRY[city]["label_dates"][-1]
    yy = dd["table"][f"class_{last_d}"].to_numpy(int)
    q = pd.qcut(dd["table"]["area"], 5, duplicates="drop", labels=False)
    ax[1].plot(range(len(np.unique(q[~pd.isna(q)]))),
               [100 * yy[q == k].mean() for k in sorted(pd.unique(q[~pd.isna(q)]))],
               marker="o", label=city)
ax[0].set_xlabel("log10 footprint area (m2)")
ax[0].set_ylabel("density")
ax[0].set_title("building size distribution")
ax[0].legend(fontsize=7)
ax[1].set_xlabel("building-size quintile (0 = smallest)")
ax[1].set_ylabel("percent damaged")
ax[1].set_title("does size predict damage, per city?")
ax[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

print("A strong prevalence trend across size quintiles indicates that "
      "footprint size carries label information in that city.")


In [ ]:
# Example post-event VV patches by city and class. Each city uses its own
# colour scale because absolute backscatter levels differ between scenes.
rng = np.random.default_rng(0)
n_ex = 3

fig, axes = plt.subplots(len(shown), 2 * n_ex,
                         figsize=(1.5 * 2 * n_ex, 1.6 * len(shown)))
axes = np.atleast_2d(axes)
for r, city in enumerate(shown):
    dd = load_city(city)
    last_d = CITY_REGISTRY[city]["label_dates"][-1]
    qi_c = dd["post_channel_names"].index("s1_post_VV")
    arr, _ = post_arrays(dd, last_d)
    okc = usable_mask(dd, last_d)
    yc = dd["table"][f"class_{last_d}"].to_numpy(int)
    idx = np.where(okc)[0]
    dmg_i = idx[yc[idx] == 1]
    int_i = idx[yc[idx] == 0]

    scale = np.asarray(arr[idx[:3000], qi_c]).astype(np.float32)
    vmin, vmax = np.percentile(scale, [2, 98])

    picks = ([rng.choice(dmg_i, min(n_ex, len(dmg_i)), replace=False)]
             + [rng.choice(int_i, min(n_ex, len(int_i)), replace=False)])
    for c, j in enumerate(np.concatenate(picks)):
        axes[r, c].imshow(np.asarray(arr[j, qi_c]).astype(np.float32),
                          cmap="gray", vmin=vmin, vmax=vmax)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title("damaged" if c < n_ex else "intact", fontsize=8)
    axes[r, 0].set_ylabel(city, fontsize=8)

plt.suptitle(f"Post-event VV, {PREP['patch_size']} px = "
             f"{PREP['patch_size'] * 10} m across", y=1.01)
plt.tight_layout()
plt.show()


## Outputs

For each successfully processed city, the notebook creates:

* `rasters/`: one pre-war S1 composite, one post composite per labelled or
  offset date, and export metadata containing the orbit and bounding box
* `processed/`: one canonical building table, one pre patch array, one post
  patch array per date, and the corresponding validity masks

Notebook 2 reads these files without an Earth Engine session. Gaza supplies
the spatial development splits, while the cities registered as holdouts are
reserved for city-level evaluation. Temporal-offset arrays provide the score
series used by the optional second-stage model.
